# FD and SPSA Toy Notebook

This notebook uses one coherent 2-dimensional noisy objective to teach five connected ideas:

1. Finite Differences (FD)
2. Simultaneous Perturbation Stochastic Approximation (SPSA)
3. Gradient estimation for black-box objectives
4. Optimization under noisy function evaluations
5. Trade-offs between accuracy, noise, and computational cost

The setup stays the same throughout:

- decision variable: $\theta = (\theta_1, \theta_2)$
- deterministic objective: a simple convex bowl-shaped function
- noisy black-box observation: the objective plus mean-zero noise
- main visual tool: the contour plot in parameter space

The key contrast is:

- **FD** = coordinate-wise perturbation
- **SPSA** = simultaneous random perturbation

## Section 1 - Problem setup

We study a very small black-box optimization problem.

Plain-English story:

- we have a parameter vector $\theta = (\theta_1, \theta_2)$
- each parameter choice produces an objective value
- lower objective values are better
- in practice we may only observe **noisy** evaluations, as in simulation-based optimization

The deterministic objective is
\[
f(\theta) = (\theta_1 - 2)^2 + 4(\theta_2 + 1)^2.
\]

The noisy observed objective is
\[
\tilde f(\theta) = f(\theta) + \varepsilon,
\]
where $\varepsilon$ is mean-zero Gaussian noise.

This is a useful toy model for black-box optimization because:
- the geometry is simple and easy to visualize
- the exact gradient is known, so we have a benchmark
- we can still pretend that we only have access to noisy function evaluations

That is exactly the situation where FD and SPSA become useful: they estimate gradients from function values rather than assuming exact derivatives are available.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('default')
np.set_printoptions(precision=4, suppress=True)

SEED = 20260424
NOISE_STD = 0.5

THETA_STAR = np.array([2.0, -1.0])
THETA0 = np.array([-3.5, 2.5], dtype=float)
THETA_PROBE = np.array([-1.0, 1.25], dtype=float)


def objective(theta):
    theta = np.asarray(theta, dtype=float)
    return (theta[..., 0] - 2.0) ** 2 + 4.0 * (theta[..., 1] + 1.0) ** 2



def noisy_objective(theta, rng, noise_std=NOISE_STD):
    return float(objective(theta) + rng.normal(loc=0.0, scale=noise_std))



def exact_gradient(theta):
    theta = np.asarray(theta, dtype=float)
    grad_1 = 2.0 * (theta[..., 0] - 2.0)
    grad_2 = 8.0 * (theta[..., 1] + 1.0)
    return np.stack([grad_1, grad_2], axis=-1)



def objective_grid(theta1_values, theta2_values):
    T1, T2 = np.meshgrid(theta1_values, theta2_values)
    grid = (T1 - 2.0) ** 2 + 4.0 * (T2 + 1.0) ** 2
    return T1, T2, grid



def finite_difference_gradient(theta, c, rng, noise_std=NOISE_STD):
    theta = np.asarray(theta, dtype=float)
    d = len(theta)
    grad = np.zeros(d, dtype=float)
    evaluation_points = []
    values = []

    for j in range(d):
        e_j = np.zeros(d)
        e_j[j] = 1.0
        theta_plus = theta + c * e_j
        theta_minus = theta - c * e_j
        f_plus = noisy_objective(theta_plus, rng, noise_std=noise_std)
        f_minus = noisy_objective(theta_minus, rng, noise_std=noise_std)
        grad[j] = (f_plus - f_minus) / (2.0 * c)
        evaluation_points.extend([theta_plus, theta_minus])
        values.extend([f_plus, f_minus])

    info = {
        "points": np.array(evaluation_points),
        "values": np.array(values),
        "eval_count": 2 * d,
    }
    return grad, info



def spsa_gradient(theta, c, rng, noise_std=NOISE_STD, delta=None):
    theta = np.asarray(theta, dtype=float)
    d = len(theta)
    if delta is None:
        delta = rng.choice([-1.0, 1.0], size=d)
    theta_plus = theta + c * delta
    theta_minus = theta - c * delta
    f_plus = noisy_objective(theta_plus, rng, noise_std=noise_std)
    f_minus = noisy_objective(theta_minus, rng, noise_std=noise_std)
    grad = (f_plus - f_minus) / (2.0 * c * delta)
    info = {
        "delta": delta,
        "theta_plus": theta_plus,
        "theta_minus": theta_minus,
        "f_plus": f_plus,
        "f_minus": f_minus,
        "eval_count": 2,
    }
    return grad, info



def step_size(a0, k, decay=0.03):
    return a0 / (1.0 + decay * k)



def perturbation_size(c0, k, decay=0.02, power=0.25):
    return c0 / (1.0 + decay * k) ** power



def run_fd_optimization(theta0, iterations, a0, c0, rng, noise_std=NOISE_STD):
    theta = np.asarray(theta0, dtype=float).copy()
    path = [theta.copy()]
    true_values = [float(objective(theta))]
    eval_counts = [0]
    grad_estimates = []

    for k in range(iterations):
        a_k = step_size(a0, k)
        c_k = perturbation_size(c0, k)
        grad_hat, info = finite_difference_gradient(theta, c_k, rng, noise_std=noise_std)
        theta = theta - a_k * grad_hat
        path.append(theta.copy())
        true_values.append(float(objective(theta)))
        eval_counts.append(eval_counts[-1] + info["eval_count"])
        grad_estimates.append(grad_hat)

    return np.array(path), np.array(true_values), np.array(eval_counts), np.array(grad_estimates)



def run_spsa_optimization(theta0, iterations, a0, c0, rng, noise_std=NOISE_STD):
    theta = np.asarray(theta0, dtype=float).copy()
    path = [theta.copy()]
    true_values = [float(objective(theta))]
    eval_counts = [0]
    grad_estimates = []

    for k in range(iterations):
        a_k = step_size(a0, k)
        c_k = perturbation_size(c0, k)
        grad_hat, info = spsa_gradient(theta, c_k, rng, noise_std=noise_std)
        theta = theta - a_k * grad_hat
        path.append(theta.copy())
        true_values.append(float(objective(theta)))
        eval_counts.append(eval_counts[-1] + info["eval_count"])
        grad_estimates.append(grad_hat)

    return np.array(path), np.array(true_values), np.array(eval_counts), np.array(grad_estimates)



def contour_levels(loss_grid):
    low = float(np.min(loss_grid))
    high = float(np.percentile(loss_grid, 97))
    if high <= low:
        high = float(np.max(loss_grid))
    return np.linspace(low, high, 18)



def normalized_arrow(vector, length=0.8):
    vector = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(vector)
    if norm < 1e-12:
        return np.zeros_like(vector)
    return length * vector / norm



def draw_path(ax, path, label, markevery=1):
    ax.plot(path[:, 0], path[:, 1], marker='o', markersize=3, linewidth=1.5, markevery=markevery, label=label)
    ax.scatter(path[0, 0], path[0, 1], marker='s', s=50)
    ax.scatter(path[-1, 0], path[-1, 1], marker='X', s=80)


print(f'True optimum theta* = {THETA_STAR}')
print(f'Exact gradient at theta* = {exact_gradient(THETA_STAR)}')
print(f'Starting point theta0 = {THETA0}')
print(f'Probe point for gradient estimation = {THETA_PROBE}')

## Section 2 - Visualize the objective landscape

The contour plot is the most important visual tool in this notebook.

It shows:
- low objective regions
- high objective regions
- the location of the optimum
- the geometry over which the optimization algorithms move

This is useful because FD and SPSA are not just formulas. They generate estimated directions in parameter space, and the contour plot lets us see those directions directly.

In [ ]:
theta1_values = np.linspace(-5.5, 5.5, 220)
theta2_values = np.linspace(-4.5, 3.5, 220)
T1, T2, loss_grid = objective_grid(theta1_values, theta2_values)
levels = contour_levels(loss_grid)

plt.figure(figsize=(8.2, 6.2))
contours = plt.contour(T1, T2, loss_grid, levels=levels)
plt.clabel(contours, inline=True, fontsize=8)
plt.scatter([THETA_STAR[0]], [THETA_STAR[1]], marker='*', s=180, label='True optimum')
plt.scatter([THETA0[0]], [THETA0[1]], marker='s', s=70, label='Optimization start')
plt.xlabel(r'Parameter $\theta_1$')
plt.ylabel(r'Parameter $\theta_2$')
plt.title('Contour plot of the true deterministic objective')
plt.legend()
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(8.2, 6.0))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(T1, T2, loss_grid, linewidth=0, antialiased=True, alpha=0.85)
ax.set_xlabel(r'$\theta_1$')
ax.set_ylabel(r'$\theta_2$')
ax.set_zlabel('Objective value')
ax.set_title('Secondary view: deterministic objective surface')
plt.tight_layout()
plt.show()

## Section 3 - Exact gradient intuition

For this toy problem, the exact gradient is known:
\[
\nabla f(\theta) =
\begin{pmatrix}
2(\theta_1 - 2) \\
8(\theta_2 + 1)
\end{pmatrix}.
\]

Plain-English interpretation:
- the gradient points uphill
- the negative gradient points downhill
- if we had exact gradients, optimization would be easier

But in many simulation-based or black-box settings, we do **not** have this formula. We may only be able to query the objective at chosen parameter points.

That is why FD and SPSA matter: they estimate gradients from function evaluations.

In [ ]:
probe_exact_grad = exact_gradient(THETA_PROBE)
opt_grad = exact_gradient(THETA_STAR)

print(f'Exact gradient at the probe point {THETA_PROBE} = {probe_exact_grad}')
print(f'Exact gradient at the optimum {THETA_STAR} = {opt_grad}')
print(f'Is the gradient near zero at the optimum? {np.allclose(opt_grad, np.zeros(2))}')

## Section 4 - Finite Differences (FD)

Finite Differences estimate the gradient by perturbing one coordinate at a time.

A two-sided FD estimate of coordinate $j$ is
\[
\hat g_j^{FD}(\theta)
=
\frac{\tilde f(\theta + c e_j) - \tilde f(\theta - c e_j)}{2c}.
\]

Plain-English meaning of the symbols:
- $e_j$ = the unit vector in coordinate $j$
- $c$ = the perturbation size
- we move a small amount forward and backward along one coordinate
- the change in objective tells us how sensitive the function is in that direction

In dimension $d$, a two-sided FD gradient estimate typically needs about $2d$ function evaluations.
So in this 2-dimensional example, FD uses 4 evaluations per gradient estimate.

In [ ]:
FD_C = 0.35
fd_rng = np.random.default_rng(SEED + 1)
fd_grad, fd_info = finite_difference_gradient(THETA_PROBE, c=FD_C, rng=fd_rng, noise_std=NOISE_STD)

print(f'FD perturbation size c = {FD_C:.3f}')
print(f'Exact gradient at probe point = {probe_exact_grad}')
print(f'One noisy FD estimate        = {fd_grad}')
print(f'FD function evaluations used = {fd_info["eval_count"]}')

fig, ax = plt.subplots(figsize=(8.2, 6.2))
ax.contour(T1, T2, loss_grid, levels=levels)
ax.scatter([THETA_PROBE[0]], [THETA_PROBE[1]], marker='o', s=80, label='Probe point')
ax.scatter(fd_info['points'][:, 0], fd_info['points'][:, 1], marker='s', s=55, label='FD perturbation points')
ax.scatter([THETA_STAR[0]], [THETA_STAR[1]], marker='*', s=180, label='True optimum')

exact_down = normalized_arrow(-probe_exact_grad, length=0.9)
fd_down = normalized_arrow(-fd_grad, length=0.9)

ax.arrow(THETA_PROBE[0], THETA_PROBE[1], exact_down[0], exact_down[1], head_width=0.08, length_includes_head=True, color='black', label='- exact gradient')
ax.arrow(THETA_PROBE[0], THETA_PROBE[1], fd_down[0], fd_down[1], head_width=0.08, length_includes_head=True, color='tab:orange', label='- FD estimate')

ax.set_xlabel(r'Parameter $\theta_1$')
ax.set_ylabel(r'Parameter $\theta_2$')
ax.set_title('FD geometry: coordinate-wise perturbation at one point')
ax.legend()
plt.tight_layout()
plt.show()

## Section 5 - SPSA

SPSA uses one random perturbation vector that changes all coordinates at the same time.

A standard SPSA estimate is
\[
\hat g_i^{SPSA}(\theta)
=
\frac{\tilde f(\theta + c\Delta) - \tilde f(\theta - c\Delta)}{2c\,\Delta_i},
\]
where each component of $\Delta$ is either $-1$ or $+1$.

Plain-English meaning:
- choose a random sign direction
- evaluate the objective at two simultaneously perturbed points
- reconstruct all gradient components from those two evaluations

The main teaching contrast is:
- FD perturbs **coordinates one by one**
- SPSA perturbs **all coordinates simultaneously**

In many dimensions this matters a lot, because SPSA still uses only 2 function evaluations per gradient estimate.

In [ ]:
SPSA_C = 0.45
spsa_rng = np.random.default_rng(SEED + 2)
spsa_grad, spsa_info = spsa_gradient(THETA_PROBE, c=SPSA_C, rng=spsa_rng, noise_std=NOISE_STD)

print(f'SPSA perturbation size c = {SPSA_C:.3f}')
print(f'SPSA random sign vector Delta = {spsa_info["delta"]}')
print(f'Exact gradient at probe point  = {probe_exact_grad}')
print(f'One noisy SPSA estimate        = {spsa_grad}')
print(f'SPSA function evaluations used = {spsa_info["eval_count"]}')

fig, ax = plt.subplots(figsize=(8.2, 6.2))
ax.contour(T1, T2, loss_grid, levels=levels)
ax.scatter([THETA_PROBE[0]], [THETA_PROBE[1]], marker='o', s=80, label='Probe point')
ax.scatter([spsa_info['theta_plus'][0], spsa_info['theta_minus'][0]], [spsa_info['theta_plus'][1], spsa_info['theta_minus'][1]], marker='s', s=55, label='SPSA perturbation points')
ax.plot([spsa_info['theta_minus'][0], spsa_info['theta_plus'][0]], [spsa_info['theta_minus'][1], spsa_info['theta_plus'][1]], linestyle='--', alpha=0.7)
ax.scatter([THETA_STAR[0]], [THETA_STAR[1]], marker='*', s=180, label='True optimum')

exact_down = normalized_arrow(-probe_exact_grad, length=0.9)
spsa_down = normalized_arrow(-spsa_grad, length=0.9)

ax.arrow(THETA_PROBE[0], THETA_PROBE[1], exact_down[0], exact_down[1], head_width=0.08, length_includes_head=True, color='black', label='- exact gradient')
ax.arrow(THETA_PROBE[0], THETA_PROBE[1], spsa_down[0], spsa_down[1], head_width=0.08, length_includes_head=True, color='tab:green', label='- SPSA estimate')

ax.set_xlabel(r'Parameter $\theta_1$')
ax.set_ylabel(r'Parameter $\theta_2$')
ax.set_title('SPSA geometry: simultaneous random perturbation at one point')
ax.legend()
plt.tight_layout()
plt.show()

## Section 6 - FD vs SPSA gradient comparison at one point

Now we compare three objects at the same probe point:
- the exact gradient
- repeated noisy FD estimates
- repeated noisy SPSA estimates

Teaching message:
- both FD and SPSA estimate the gradient
- both fluctuate under noise
- FD may be more stable in low dimensions
- SPSA is cheaper in function evaluations, but each estimate can be noisier

In [ ]:
REPEATS = 25
fd_estimates = []
spsa_estimates = []

fd_repeat_rng = np.random.default_rng(SEED + 10)
spsa_repeat_rng = np.random.default_rng(SEED + 20)

for _ in range(REPEATS):
    fd_estimate, _ = finite_difference_gradient(THETA_PROBE, c=FD_C, rng=fd_repeat_rng, noise_std=NOISE_STD)
    spsa_estimate, _ = spsa_gradient(THETA_PROBE, c=SPSA_C, rng=spsa_repeat_rng, noise_std=NOISE_STD)
    fd_estimates.append(fd_estimate)
    spsa_estimates.append(spsa_estimate)

fd_estimates = np.array(fd_estimates)
spsa_estimates = np.array(spsa_estimates)

print('Exact gradient at probe point:', probe_exact_grad)
print('FD estimate mean over repeats:', fd_estimates.mean(axis=0))
print('SPSA estimate mean over repeats:', spsa_estimates.mean(axis=0))
print('FD estimate std over repeats:', fd_estimates.std(axis=0))
print('SPSA estimate std over repeats:', spsa_estimates.std(axis=0))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
iteration_index = np.arange(1, REPEATS + 1)

axes[0].plot(iteration_index, fd_estimates[:, 0], marker='o', linestyle='None', label='FD estimates')
axes[0].plot(iteration_index, spsa_estimates[:, 0], marker='x', linestyle='None', label='SPSA estimates')
axes[0].axhline(probe_exact_grad[0], linestyle='--', color='black', label='Exact gradient')
axes[0].set_title(r'Gradient component $g_1$ at one point')
axes[0].set_xlabel('Repeated noisy estimates')
axes[0].set_ylabel('Estimated value')
axes[0].legend()

axes[1].plot(iteration_index, fd_estimates[:, 1], marker='o', linestyle='None', label='FD estimates')
axes[1].plot(iteration_index, spsa_estimates[:, 1], marker='x', linestyle='None', label='SPSA estimates')
axes[1].axhline(probe_exact_grad[1], linestyle='--', color='black', label='Exact gradient')
axes[1].set_title(r'Gradient component $g_2$ at one point')
axes[1].set_xlabel('Repeated noisy estimates')
axes[1].legend()

plt.tight_layout()
plt.show()

## Section 7 - Optimization using FD

Now we place the FD gradient estimate inside an iterative optimization loop:
\[
\theta_{k+1} = \theta_k - a_k \hat g_k.
\]

Here:
- $\hat g_k$ is the FD gradient estimate at iteration $k$
- $a_k$ is the step size

The path will move through the contour plot using noisy coordinate-wise gradient estimates.

In [ ]:
FD_ITERS = 35
FD_A0 = 0.10
FD_C0 = 0.35

fd_path, fd_true_values, fd_eval_counts, fd_grad_history = run_fd_optimization(
    theta0=THETA0,
    iterations=FD_ITERS,
    a0=FD_A0,
    c0=FD_C0,
    rng=np.random.default_rng(SEED + 30),
    noise_std=NOISE_STD,
)

print(f'FD optimization final theta = {fd_path[-1]}')
print(f'FD optimization final true objective = {fd_true_values[-1]:.6f}')
print(f'FD total function evaluations = {fd_eval_counts[-1]}')

fig, ax = plt.subplots(figsize=(8.2, 6.2))
ax.contour(T1, T2, loss_grid, levels=levels)
draw_path(ax, fd_path, label='FD optimization path', markevery=2)
ax.scatter([THETA_STAR[0]], [THETA_STAR[1]], marker='*', s=180, label='True optimum')
ax.set_xlabel(r'Parameter $\theta_1$')
ax.set_ylabel(r'Parameter $\theta_2$')
ax.set_title('FD-based optimization path on the contour map')
ax.legend()
plt.tight_layout()
plt.show()

## Section 8 - Optimization using SPSA

SPSA uses the same basic update idea,
\[
\theta_{k+1} = \theta_k - a_k \hat g_k,
\]
but now the gradient estimate comes from simultaneous random perturbation.

This usually makes each gradient estimate cheaper in function evaluations, though noisier.

In [ ]:
SPSA_ITERS = 35
SPSA_A0 = 0.05
SPSA_C0 = 0.45

spsa_path, spsa_true_values, spsa_eval_counts, spsa_grad_history = run_spsa_optimization(
    theta0=THETA0,
    iterations=SPSA_ITERS,
    a0=SPSA_A0,
    c0=SPSA_C0,
    rng=np.random.default_rng(SEED + 40),
    noise_std=NOISE_STD,
)

print(f'SPSA optimization final theta = {spsa_path[-1]}')
print(f'SPSA optimization final true objective = {spsa_true_values[-1]:.6f}')
print(f'SPSA total function evaluations = {spsa_eval_counts[-1]}')

fig, ax = plt.subplots(figsize=(8.2, 6.2))
ax.contour(T1, T2, loss_grid, levels=levels)
draw_path(ax, spsa_path, label='SPSA optimization path', markevery=2)
ax.scatter([THETA_STAR[0]], [THETA_STAR[1]], marker='*', s=180, label='True optimum')
ax.set_xlabel(r'Parameter $\theta_1$')
ax.set_ylabel(r'Parameter $\theta_2$')
ax.set_title('SPSA-based optimization path on the contour map')
ax.legend()
plt.tight_layout()
plt.show()

## Section 9 - Compare optimization behavior

Now we compare FD-based and SPSA-based optimization in three ways:
1. path toward the optimum
2. objective value versus iteration
3. objective value versus function evaluations

This third comparison is especially important.
Iteration count alone does **not** tell the whole story, because FD and SPSA spend different numbers of function evaluations per gradient estimate.

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 6.2))
ax.contour(T1, T2, loss_grid, levels=levels)
draw_path(ax, fd_path, label='FD path', markevery=2)
draw_path(ax, spsa_path, label='SPSA path', markevery=2)
ax.scatter([THETA_STAR[0]], [THETA_STAR[1]], marker='*', s=180, label='True optimum')
ax.set_xlabel(r'Parameter $\theta_1$')
ax.set_ylabel(r'Parameter $\theta_2$')
ax.set_title('FD vs SPSA optimization paths on the same contour plot')
ax.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.8))
plt.plot(fd_true_values, label='FD true objective')
plt.plot(spsa_true_values, label='SPSA true objective')
plt.xlabel('Optimization iterations')
plt.ylabel('True deterministic objective value')
plt.title('Objective value versus iteration')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.8))
plt.plot(fd_eval_counts, fd_true_values, label='FD')
plt.plot(spsa_eval_counts, spsa_true_values, label='SPSA')
plt.xlabel('Function evaluations')
plt.ylabel('True deterministic objective value')
plt.title('Objective value versus function evaluations')
plt.legend()
plt.tight_layout()
plt.show()

print(f'FD total evaluations after {FD_ITERS} iterations   = {fd_eval_counts[-1]}')
print(f'SPSA total evaluations after {SPSA_ITERS} iterations = {spsa_eval_counts[-1]}')

## Section 10 - Dimensionality thought experiment

A major reason SPSA is attractive is that function-evaluation cost scales very differently with dimension.

For a two-sided FD estimate in dimension $d$:
- about $2d$ function evaluations are needed per gradient estimate

For SPSA:
- typically 2 function evaluations are needed per gradient estimate, regardless of dimension

This does **not** mean SPSA is magic.
It means that SPSA trades cheaper evaluations for noisier simultaneous perturbation estimates.

In [ ]:
dimensions = np.array([2, 5, 20])
fd_costs = 2 * dimensions
spsa_costs = np.full_like(dimensions, 2)

for d, fd_cost, spsa_cost in zip(dimensions, fd_costs, spsa_costs):
    print(f'd = {d:2d}: FD two-sided cost = {fd_cost:2d} evaluations, SPSA cost = {spsa_cost:2d} evaluations')

plt.figure(figsize=(7.8, 4.8))
width = 0.35
xpos = np.arange(len(dimensions))
plt.bar(xpos - width / 2, fd_costs, width=width, label='FD two-sided evaluations')
plt.bar(xpos + width / 2, spsa_costs, width=width, label='SPSA evaluations')
plt.xticks(xpos, [str(d) for d in dimensions])
plt.xlabel('Dimension d')
plt.ylabel('Function evaluations per gradient estimate')
plt.title('Evaluation cost scaling with dimension')
plt.legend()
plt.tight_layout()
plt.show()

## Section 11 - Connect the ideas

| Concept | Role | Plain-English meaning |
|---|---|---|
| Exact gradient | Ideal benchmark | The true local uphill direction of the deterministic objective. |
| FD gradient estimate | Coordinate-wise approximation | Estimate each gradient component by perturbing one coordinate at a time. |
| SPSA gradient estimate | Simultaneous random approximation | Estimate all gradient components from two simultaneous random perturbations. |
| Perturbation cost | Evaluation budget | How many objective evaluations are needed to build one gradient estimate. |
| Noise behavior | Estimate variability | Both FD and SPSA fluctuate under noisy function evaluations. |
| Best use case | Practical decision guide | FD is intuitive and often strong in low dimensions; SPSA becomes attractive when evaluations are expensive and dimension grows. |

## Section 12 - Final takeaway

The main lessons are:

- FD and SPSA both estimate gradients when exact gradients are unavailable
- FD perturbs coordinates one by one
- SPSA perturbs all coordinates simultaneously
- FD is intuitive and often strong in low dimensions
- SPSA is especially attractive when function evaluations are expensive and dimension grows
- both are central tools in simulation-based optimization

A final distinction to remember:
- the **exact gradient** is the true derivative of the deterministic objective
- the **estimated gradient** is an approximation built from noisy function evaluations
- the **optimization step** is what we do with that estimated gradient
- the **function-evaluation cost** measures how expensive it is to build the estimate in the first place

So the main teaching contrast is not "good" versus "bad".
It is:

> coordinate-wise perturbation versus simultaneous random perturbation.

## Teaching notes

### Likely misunderstandings

- **"FD is bad because it uses more evaluations."**
  Not at all. In low dimensions FD can be intuitive, accurate, and very competitive.

- **"SPSA is magic because it uses only two evaluations."**
  No. SPSA gets that efficiency by accepting noisier simultaneous perturbation estimates.

- **"A noisy gradient estimate is useless."**
  Not necessarily. A noisy estimate can still be informative enough to guide optimization.

- **"Iteration count alone tells us which optimizer is better."**
  Not here. Function-evaluation cost is one of the main teaching points, so objective-versus-evaluations matters a lot.

- **"The optimization step and the gradient estimate are the same object."**
  They are different. First we estimate a gradient, then we choose a step size, then we update the parameter.

### What each plot is intended to teach

- **Main contour plot:** the optimization problem has geometry, not just numbers.
- **FD geometry plot:** FD perturbs one coordinate at a time and uses those directional changes to estimate the gradient.
- **SPSA geometry plot:** SPSA perturbs all coordinates simultaneously using a random sign vector.
- **Repeated-estimate plot:** both FD and SPSA fluctuate under noise, but in different ways.
- **Optimization-path plots:** gradient estimates become actual iterative movement through parameter space.
- **Objective-versus-evaluations plot:** evaluation efficiency is a core practical comparison, not an afterthought.
- **Dimensionality plot:** SPSA's evaluation cost scales very differently from FD as dimension grows.